# Исследование территории города Гусев (Калининградская область)

Ноутбук подготовлен для учебного исследования. Он пошагово:

1. определяет границу территории;
2. готовит **кварталы** (наименьшие единицы анализа) по данным OpenStreetMap (далее — OSM);
3. скачивает **готовый пешеходный граф** и считает **время в пути между кварталами в минутах**;
4. собирает данные о **зданиях и населении** и о **сервисах** (по умолчанию — школы);
5. считает **показатели** — те же, что в примере `blocksnet/examples/pipeline.ipynb`;
6. сохраняет все данные и результаты в **GeoJSON/CSV**, чтобы их можно было переиспользовать.

## Источники данных и допущения

| Что | Источник | Примечание |
|---|---|---|
| Граница территории | OpenStreetMap (OSM), **полигон города Гусев** (relation 19338643, `place=town`) | город ≈ 21 км², население ≈ 28,5 тыс. чел. |
| Вода, дороги, железные дороги, здания, функциональные зоны, школы | OSM (загрузка через `osmnx`) | лицензия ODbL, © участники OpenStreetMap |
| Граф и время в пути | готовый **пешеходный** граф OSM (скачивается через `blocksnet`/`iduedu`/`osmnx`) | скорость **5 км/ч**, время в **минутах** |

**Что такое «территория» здесь.** Исследуемая территория — **полигон города Гусев** из OSM (relation 19338643, `place=town`). Это компактная городская территория площадью ≈ **21 км²** (около 5,5 × 7,5 км), то есть сам город, без окрестных посёлков и полей. Граница берётся готовой из OSM, ничего обрезать и достраивать не нужно.

Если позже понадобится более широкая территория (город + округа), можно подложить свой полигон в файл `data/boundaries.geojson` — ноутбук подхватит его вместо границы из OSM.

## Библиотеки

* `blocksnet==1.0.0a9` — расчёт кварталов, доступности и показателей;
* `iduedu==0.4.1` — городские графы (используется внутри blocksnet);
* `osmnx==2.1.1` — загрузка данных OSM;
* `geopandas`, `pandas`, `matplotlib`.

## Как запускать

* Ноутбук рассчитан на последовательный запуск ячеек сверху вниз (кнопка **Run All** / «Запустить все»).
* **Первый запуск самый долгий**: скачивание объектов, зданий и пешеходного графа из OSM для города обычно занимает 3–10 минут (зависит от интернета и загрузки серверов OSM). Повторные запуски читают готовые файлы из `data/` и занимают меньше минуты.
* Правило переиспользования: **если файл уже есть — читаем его; если файла нет — считаем и сохраняем**. Удалить файлы, чтобы скачать заново, можно функцией `clear_cache()` (см. последний раздел).
* Ячейка **«СТОП»** (часть 5) — информационная: на этом месте мы обсуждаем, какие показатели считать дополнительно. Она ничего не блокирует.

## 0. Настройка окружения и правила работы с файлами

### 0.1. Настройки исследования

Здесь задаются папки и параметры. **Это единственная ячейка, куда обычно нужно заглядывать.**

* `OSM_RELATION_ID = 19338643` — номер объекта OSM, из которого берётся граница города (менять не нужно).
* `CITY_NAME` — название города только для подписей в тексте и на картах.
* `WALK_SPEED_KMH` — скорость пешехода для контроля (граф скачивается со скоростью по умолчанию 5 км/ч).

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd

warnings.filterwarnings("ignore")  # меньше служебных предупреждений библиотек
pd.set_option("display.max_columns", 60)


# --- папки: ноутбук работает и при запуске из VS Code, и из корня репозитория ---
def find_notebook_dir() -> Path:
    here = Path.cwd().resolve()
    if here.name == "gusev":
        return here
    candidate = here / "examples" / "gusev"
    if candidate.exists():
        return candidate
    return here


NOTEBOOK_DIR = find_notebook_dir()
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- параметры исследования (можно менять) ---
CITY_NAME = "Гусев"  # название города (для подписей)
OSM_RELATION_ID = 19338643  # OSM: город Гусев (place=town) — отсюда берётся граница города
WALK_SPEED_KMH = 5  # км/ч — скорость пешехода в графе
DEFAULT_LIVING_DEMAND = 30  # м² жилой площади на человека (для оценки населения)

# --- кеш OSM: повторные загрузки тех же объектов будут мгновенными ---
ox.settings.use_cache = True
ox.settings.cache_folder = DATA_DIR / "osm_cache"
ox.settings.log_console = False

print("Папка ноутбука:", NOTEBOOK_DIR)
print("Папка данных:  ", DATA_DIR)
print(f"Город: {CITY_NAME} | OSM relation: {OSM_RELATION_ID} | скорость пешехода: {WALK_SPEED_KMH} км/ч")

### 0.2. Функции сохранения, загрузки и очистки кеша

Все результаты складываются в папку `data` рядом с ноутбуком:

| Файл | Что внутри |
|---|---|
| `boundaries.geojson` | граница исследуемой территории |
| `water/roads/railways.geojson` | вода, дороги, железные дороги из OSM |
| `lines/polygons.geojson` | геометрии, подготовленные для нарезки кварталов |
| `functional_zones.geojson` | зоны функционального использования (landuse) из OSM |
| `blocks_land_use.geojson` | кварталы с долями land use |
| `walk_graph.pkl` | пешеходный граф (для расчёта времени в пути) |
| `walk_nodes/walk_edges.geojson` | узлы и рёбра графа (для просмотра в QGIS и на карте) |
| `accessibility_matrix_walk.csv` | матрица времени в пути между кварталами, **минуты** |
| `buildings_blocks.geojson` | здания, агрегированные по кварталам (население и площади) |
| `schools_blocks.geojson` | школы, агрегированные по кварталам (ёмкость) |
| `density_indicators.csv`, `spacematrix_*.csv`, `strelka_morphotypes.csv` | показатели плотности и морфотипы |
| `provision_school.csv`, `links_school.csv` | обеспеченность школами и связи «спрос → ёмкость» |
| `area_accessibility.csv` | доступность территории |
| `blocks_indicators.geojson` | **итог: кварталы со всеми показателями** |

In [ ]:
def _geojson_ready(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Готовит слой к записи в GeoJSON: перечисления (например LandUse) -> строки."""
    gdf = gdf.copy()
    geom_col = gdf.geometry.name
    for col in gdf.columns:
        if col == geom_col:
            continue
        if gdf[col].dtype == object or isinstance(gdf[col].dtype, pd.CategoricalDtype):

            def to_plain(value):
                if hasattr(value, "value"):  # перечисления blocksnet
                    return value.value
                if value is None or isinstance(value, (bool, int, float, str)):
                    return value
                return str(value)

            gdf[col] = gdf[col].map(to_plain)
    return gdf


def save_gdf(gdf: gpd.GeoDataFrame, filename: str) -> Path:
    """Сохраняет слой в GeoJSON в папке data."""
    out = _geojson_ready(gdf)
    if isinstance(out.index, pd.MultiIndex):
        out = out.reset_index()
    elif out.index.name:
        out = out.reset_index()
    else:
        out = out.reset_index(drop=True)
    path = DATA_DIR / filename
    out.to_file(path, driver="GeoJSON", encoding="utf-8")
    print(f"[сохранено] {filename}: объектов {len(out)}")
    return path


def load_gdf(filename: str) -> gpd.GeoDataFrame:
    """Читает слой из папки data; идентификатор квартала снова становится индексом."""
    gdf = gpd.read_file(DATA_DIR / filename)
    if "block_id" in gdf.columns:
        gdf = gdf.set_index("block_id").sort_index()
    return gdf


def has_file(filename: str) -> bool:
    return (DATA_DIR / filename).exists()


def save_df(df: pd.DataFrame, filename: str, index: bool = True) -> Path:
    path = DATA_DIR / filename
    df.to_csv(path, index=index)
    print(f"[сохранено] {filename}: строк {len(df)}, столбцов {len(df.columns)}")
    return path


def load_df(filename: str) -> pd.DataFrame:
    """Читает CSV; числовые индексы/столбцы возвращает в виде чисел."""
    df = pd.read_csv(DATA_DIR / filename, index_col=0)
    if all(str(value).lstrip("-").isdigit() for value in df.index):
        df.index = df.index.astype(int)
    if all(str(value).lstrip("-").isdigit() for value in df.columns):
        df.columns = df.columns.astype(int)
    return df


def save_pickle(obj, filename: str) -> Path:
    path = DATA_DIR / filename
    with open(path, "wb") as file:
        pickle.dump(obj, file)
    print(f"[сохранено] {filename}")
    return path


def load_pickle(filename: str):
    with open(DATA_DIR / filename, "rb") as file:
        return pickle.load(file)


def list_data_files() -> pd.DataFrame:
    """Список файлов в папке data с размерами."""
    rows = []
    for path in sorted(DATA_DIR.rglob("*")):
        if path.is_file():
            rows.append({"файл": str(path.relative_to(DATA_DIR)), "размер, КБ": round(path.stat().st_size / 1024, 1)})
    return pd.DataFrame(rows)


def clear_cache(include_osm_cache: bool = True) -> None:
    """Удаляет сохранённые файлы. После этого данные будут скачаны заново."""
    removed = 0
    for path in sorted(DATA_DIR.rglob("*"), reverse=True):
        if path.is_file() and path.name != ".gitkeep":
            if not include_osm_cache and "osm_cache" in path.parts:
                continue
            path.unlink()
            removed += 1
    print(f"[кеш очищен] удалено файлов: {removed}")


def check(name: str, condition: bool, details: str = "") -> None:
    """Печать результата проверки: [ОК] или [ПРОБЛЕМА]."""
    mark = "[ОК]      " if condition else "[ПРОБЛЕМА]"
    print(f"{mark} {name}" + (f" — {details}" if details else ""))


for known_file in ["boundaries.geojson", "blocks.geojson", "walk_graph.pkl", "blocks_indicators.geojson"]:
    print(f"{known_file:<30} {'уже есть' if has_file(known_file) else 'пока нет'}")

## 1. Граница территории

Берём границу города из OSM: полигон **Гусева** (relation 19338643, `place=town`), площадь ≈ 21 км². Это и есть территория исследования — сам город.

Если файл `data/boundaries.geojson` уже есть, скачивание не выполняется (а если положить в него свой полигон, ноутбук возьмёт его вместо границы из OSM).

In [ ]:
def download_boundaries() -> gpd.GeoDataFrame:
    """Полигон города из OSM по номеру объекта."""
    gdf = ox.geocode_to_gdf(f"R{OSM_RELATION_ID}", by_osmid=True).to_crs(4326)
    gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index(drop=True)
    if gdf.empty:
        raise RuntimeError(
            "OSM не отдал полигон города для OSM_RELATION_ID. "
            "Положите свой файл границы в data/boundaries.geojson и запустите ячейку снова."
        )
    return gdf


def build_territory() -> gpd.GeoDataFrame:
    """Граница исследуемой территории в метрах."""
    boundaries_4326 = download_boundaries()
    crs = boundaries_4326.estimate_utm_crs()
    territory = boundaries_4326.to_crs(crs)
    print(f"Граница города получена из OSM (relation {OSM_RELATION_ID})")
    return territory[["geometry"]].reset_index(drop=True)


if has_file("boundaries.geojson"):
    boundaries = load_gdf("boundaries.geojson")
    print("Граница прочитана из файла boundaries.geojson (повторный запуск — OSM не скачивается)")
else:
    boundaries = build_territory()
    save_gdf(boundaries, "boundaries.geojson")

print("Тип геометрии:", ", ".join(sorted(boundaries.geom_type.unique())))
print("Система координат (CRS): EPSG:", boundaries.crs.to_epsg())

In [ ]:
area_km2 = boundaries.area.sum() / 1e6
print(f"Площадь исследуемой территории: {area_km2:.1f} км²")

ax = boundaries.plot(color="#eef2f7", edgecolor="#33415c", linewidth=1.5, figsize=(7, 7))
ax.set_title("Граница исследуемой территории")
ax.set_axis_off()
plt.show()

## 2. Исходные объекты OpenStreetMap: вода, дороги, железные дороги

Для нарезки кварталов нужны «барьеры»: дороги, железные дороги и вода. Мы скачиваем их из OSM по территории из предыдущего шага. Набор тегов такой же, как в примере `pipeline.ipynb`.

**Внимание:** здесь происходит загрузка из интернета — первый раз может занять несколько минут.

In [ ]:
BC_TAGS = {
    "roads": {
        "highway": [
            "construction", "crossing", "living_street", "motorway", "motorway_link", "motorway_junction",
            "pedestrian", "primary", "primary_link", "raceway", "residential", "road", "secondary",
            "secondary_link", "services", "tertiary", "tertiary_link", "track", "trunk", "trunk_link",
            "turning_circle", "turning_loop", "unclassified",
        ],
        "service": ["living_street", "emergency_access"],
    },
    "railways": {"railway": "rail"},
    "water": {
        "riverbank": True,
        "reservoir": True,
        "basin": True,
        "dock": True,
        "canal": True,
        "pond": True,
        "natural": ["water", "bay"],
        "waterway": ["river", "canal", "ditch"],
        "landuse": "basin",
        "water": "lake",
    },
}


def download_features(tags: dict, filename: str, geometry_types: list) -> gpd.GeoDataFrame:
    """Скачивает объекты OSM (или читает сохранённый файл) и оставляет нужные типы геометрии."""
    if has_file(filename):
        gdf = load_gdf(filename)
        print(f"[из файла] {filename}: объектов {len(gdf)}")
        return gdf
    print(f"[скачиваем из OSM] {filename} ...")
    polygon_4326 = boundaries.to_crs(4326).union_all()
    gdf = ox.features_from_polygon(polygon_4326, tags=tags)
    gdf = gdf[gdf.geom_type.isin(geometry_types)].reset_index(drop=True).to_crs(boundaries.crs)
    save_gdf(gdf, filename)
    return gdf


water = download_features(
    BC_TAGS["water"], "water.geojson", ["Polygon", "MultiPolygon", "LineString", "MultiLineString"]
)
roads = download_features(BC_TAGS["roads"], "roads.geojson", ["LineString", "MultiLineString"])
railways = download_features(BC_TAGS["railways"], "railways.geojson", ["LineString", "MultiLineString"])

print(f"Вода: {len(water)} объектов | дороги: {len(roads)} | железные дороги: {len(railways)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
boundaries.plot(ax=ax, color="#f7f7f7", edgecolor="#999999", linewidth=1)
water.plot(ax=ax, color="#a6c8e0")
roads.plot(ax=ax, color="#e07a5f", linewidth=0.7)
railways.plot(ax=ax, color="#3d405b", linewidth=1.2, linestyle="--")
ax.set_title("Исходные объекты: вода, дороги, железные дороги")
ax.set_axis_off()
plt.show()

## 3. Подготовка кварталов

**Квартал** — это участок территории, ограниченный дорогами, железными дорогами и водой. Именно кварталы в дальнейшем выступают «точками» анализа: по ним считаются население, сервисы, показатели и время в пути.

Шаги внутри:

1. `preprocess_urban_objects` — приводит геометрии к единому виду и разделяет их на линии и полигоны;
2. `cut_urban_blocks` — «режет» территорию на кварталы.

Крупные водоёмы (реки, пруды, озёра) являются «вырезаемыми» объектами, поэтому кварталы создаются только на суше.

In [ ]:
from blocksnet.blocks.cutting import cut_urban_blocks, preprocess_urban_objects

if has_file("lines.geojson") and has_file("polygons.geojson"):
    lines = load_gdf("lines.geojson")
    polygons = load_gdf("polygons.geojson")
    print(f"Геометрии прочитаны из файлов (линий: {len(lines)}, полигонов: {len(polygons)})")
else:
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    save_gdf(lines, "lines.geojson")
    save_gdf(polygons, "polygons.geojson")

print(f"Линий для нарезки: {len(lines)} | полигонов (барьеров): {len(polygons)}")

In [ ]:
if has_file("blocks.geojson"):
    blocks = load_gdf("blocks.geojson")
    print(f"Кварталы прочитаны из файла blocks.geojson: {len(blocks)}")
else:
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    blocks = blocks.reset_index(drop=True)
    blocks.index.name = "block_id"
    save_gdf(blocks, "blocks.geojson")

blocks.index.name = "block_id"

print(f"Кварталов: {len(blocks)}")
print(f"Суммарная площадь кварталов: {blocks.area.sum() / 1e6:.1f} км²")
print(f"Медианная площадь квартала: {blocks.area.median() / 1e4:.2f} га")

ax = blocks.plot(color="#dfe7ef", edgecolor="#5c6b7a", linewidth=0.4, figsize=(9, 9))
ax.set_title(f"Кварталы ({len(blocks)} шт.)")
ax.set_axis_off()
plt.show()

## 4. Функциональные зоны и назначение land use (тип использования территории)

Скачиваем из OSM полигоны `landuse` (жилые, промышленные, рекреационные и т.д.), сопоставляем их с категориями `LandUse` библиотеки и для каждого квартала считаем **доли** каждого типа использования, а также выбираем преобладающий тип (`land_use`).

In [ ]:
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.enums import LandUse

RULES = {
    "commercial": LandUse.BUSINESS,
    "industrial": LandUse.INDUSTRIAL,
    "cemetery": LandUse.SPECIAL,
    "garages": LandUse.INDUSTRIAL,
    "residential": LandUse.RESIDENTIAL,
    "retail": LandUse.BUSINESS,
    "grass": LandUse.RECREATION,
    "farmland": LandUse.AGRICULTURE,
    "construction": LandUse.SPECIAL,
    "brownfield": LandUse.INDUSTRIAL,
    "forest": LandUse.RECREATION,
    "recreation_ground": LandUse.RECREATION,
    "religious": LandUse.SPECIAL,
    "flowerbed": LandUse.RECREATION,
    "military": LandUse.SPECIAL,
    "landfill": LandUse.TRANSPORT,
}

if has_file("functional_zones.geojson"):
    functional_zones = load_gdf("functional_zones.geojson")
    print(f"Функциональные зоны прочитаны из файла: {len(functional_zones)} полигонов")
else:
    polygon_4326 = boundaries.to_crs(4326).union_all()
    try:
        zones = ox.features_from_polygon(polygon_4326, tags={"landuse": True})
    except Exception as error:  # обычно это сеть или пустой ответ OSM
        raise RuntimeError(
            f"Не удалось скачать функциональные зоны (landuse) из OSM: {error}. Запустите ячейку снова."
        ) from error
    if "landuse" not in zones.columns:
        raise RuntimeError("В OSM нет зон landuse на этой территории — проверьте границу в data/boundaries.geojson.")
    zones = (
        zones.reset_index(drop=True)[["geometry", "landuse"]]
        .rename(columns={"landuse": "functional_zone"})
        .to_crs(boundaries.crs)
    )
    functional_zones = zones[zones.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index(drop=True)
    save_gdf(functional_zones, "functional_zones.geojson")

print("Типы функциональных зон:", sorted(functional_zones["functional_zone"].dropna().unique().tolist()))

In [ ]:
if has_file("blocks_land_use.geojson"):
    blocks_land_use = load_gdf("blocks_land_use.geojson")
    print(f"Слой кварталов с land use прочитан из файла: {len(blocks_land_use)}")
else:
    blocks_land_use = assign_land_use(blocks, functional_zones, RULES)
    blocks_land_use.index.name = "block_id"
    save_gdf(blocks_land_use, "blocks_land_use.geojson")

share_columns = [land_use.value for land_use in LandUse]
share_columns = [column for column in share_columns if column in blocks_land_use.columns]
print("Столбцы с долями типов использования:", share_columns)
print("Кварталов с определённым land use:", int(blocks_land_use["land_use"].notna().sum()), "из", len(blocks_land_use))

In [ ]:
use_counts = blocks_land_use["land_use"].fillna("не определён").astype(str).value_counts()
print("Преобладающий тип использования по кварталам:")
print(use_counts.to_string())

fig, ax = plt.subplots(figsize=(9, 9))
axis = blocks_land_use.plot(
    column="land_use", categorical=True, legend=True, ax=ax, cmap="tab10",
    edgecolor="#7a7a7a", linewidth=0.4, missing_kwds={"color": "#e6e6e6"},
)
axis.set_title("Land use (преобладающий тип использования квартала)")
axis.set_axis_off()
plt.show()

if share_columns:
    mean_shares = (blocks_land_use[share_columns].mean() * 100).round(1).sort_values(ascending=False)
    print("Средняя доля типа использования по территории, %:")
    print(mean_shares.to_string())

## 5. СТОП: данные и кварталы готовы — перед расчётами

На этом месте **остановитесь и напишите мне**.

Уже сделано: граница → исходные объекты OSM → кварталы → land use. Дальше начинаются расчёты (граф, время в пути, показатели), которые зависят от того, **какие метрики вам нужны**.

Что уже запланировано в ноутбуке (как в `pipeline.ipynb`):

* время в пути между кварталами по **пешеходному** графу (минуты);
* население по кварталам (из зданий);
* сервисы: **школы** — ёмкость по кварталам;
* показатели: обеспеченность школами (`competitive_provision`), плотностные показатели (FSI, GSI, MXI, L, OSR), морфотипы Spacematrix и Strelka, доступность территории (`area_accessibility`).

Что можно добавить (обсудим):

* другие сервисы из конфигурации blocksnet: детские сады, поликлиники, аптеки, магазины, спортплощадки, парки и др.;
* медианную / максимальную доступность, связность, центральность (по населению и сервисам);
* разнообразие land use (индекс Шеннона), пустующие территории;
* набор социально-экономических индикаторов blocksnet (генплан, демография, транспорт и др.).

**Ниже** — ячейка с краткой сводкой текущего состояния. После неё начинаются расчёты.

Территория исследования — город Гусев (полигон из OSM, ≈ 21 км²); кварталы создаются только на суше, крупные водоёмы при нарезке вырезаются.

In [ ]:
print("=== СТОП: подготовка завершена ===")
print(f"Территория: {CITY_NAME} | площадь {boundaries.area.sum() / 1e6:.1f} км²")
print(f"Кварталов: {len(blocks)}")
print(f"С land use: {int(blocks_land_use['land_use'].notna().sum())} из {len(blocks_land_use)}")
print("Напишите мне — я уточню, какие метрики добавить, и мы продолжим расчёты.")

## 6. Пешеходный граф (готовый, скачивается из OpenStreetMap)

Для расчёта времени в пути берём **готовый пешеходный граф** — его скачивает библиотека (тип графа `"walk"`, интермодальный граф не строим).

* **Источник:** OpenStreetMap (© участники OpenStreetMap, лицензия ODbL), загрузка через `osmnx`/`iduedu` внутри `blocksnet`.
* **Способ передвижения:** пешком, **5 км/ч** (скорость по умолчанию в библиотеке).
* **Единицы:** длина рёбер в метрах (`length_meter`), время в **минутах** (`time_min`).
* Граф покрывает выпуклую оболочку территории, поэтому выходит за её границы (в том числе над водой) — это нормально. Библиотека сама оставляет только основную связную часть сети, а оторванные участки (например, на островах) убирает.

Граф сохраняется в `data/walk_graph.pkl` (для расчётов) и в `walk_nodes/walk_edges.geojson` (для просмотра).

**Внимание:** скачивание графа — самая долгая операция при первом запуске (для города обычно это несколько минут).

In [ ]:
from blocksnet.relations import accessibility_graph_to_gdfs, get_accessibility_graph

if has_file("walk_graph.pkl"):
    graph = load_pickle("walk_graph.pkl")
    print("Граф прочитан из файла walk_graph.pkl")
else:
    print("Скачиваем пешеходный граф из OpenStreetMap... это самая долгая операция, подождите")
    graph = get_accessibility_graph(boundaries, "walk")
    save_pickle(graph, "walk_graph.pkl")

print("Тип графа:      ", graph.graph.get("type"))
print("CRS (EPSG):     ", graph.graph.get("crs"))
print("Скорость в графе:", graph.graph.get("walk_speed"), "м/мин (5 км/ч)")
print("Узлов:", graph.number_of_nodes(), "| рёбер:", graph.number_of_edges())

In [ ]:
if has_file("walk_nodes.geojson") and has_file("walk_edges.geojson"):
    walk_nodes = load_gdf("walk_nodes.geojson")
    walk_edges = load_gdf("walk_edges.geojson")
    print("Узлы и рёбра прочитаны из файлов")
else:
    walk_nodes, walk_edges = accessibility_graph_to_gdfs(graph)
    walk_nodes.index.name = "node_id"
    walk_edges.index = walk_edges.index.set_names(["u", "v"])
    save_gdf(walk_nodes, "walk_nodes.geojson")
    save_gdf(walk_edges, "walk_edges.geojson")

print("Пример рёбер графа (время в минутах — столбец time_min):")
walk_edges[["length_meter", "time_min"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
blocks.plot(ax=ax, color="#f2f2f2", edgecolor="#bbbbbb", linewidth=0.3)
walk_edges.plot(ax=ax, color="#4a7fb5", linewidth=0.5)
ax.set_title("Пешеходный граф поверх кварталов")
ax.set_axis_off()
plt.show()

## 7. Время в пути между кварталами (минуты)

`calculate_accessibility_matrix` считает кратчайшее время в пути **пешком** от квартала до квартала по графу. Значения — **в минутах**; используется вес рёбер `time_min`.

Как это работает: библиотека сама «подключает» каждый квартал к ближайшему узлу пешеходного графа, поэтому кварталы на краю территории тоже получают время в пути (оно будет больше). Если маршрута нет вовсе, в матрице будет `inf` («недостижимо»). Ниже мы посмотрим, сколько таких пар и какие кварталы «отрезаны».

In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

if has_file("accessibility_matrix_walk.csv"):
    acc_mx = load_df("accessibility_matrix_walk.csv")
    print(f"Матрица прочитана из файла: {acc_mx.shape[0]} × {acc_mx.shape[1]}")
else:
    print("Считаем матрицу времени в пути (пешком)...")
    acc_mx = calculate_accessibility_matrix(blocks, graph, weight_key="time_min", dtype=np.float32)
    acc_mx.index.name = "block_id"
    save_df(acc_mx, "accessibility_matrix_walk.csv")

print("Матрица:", acc_mx.shape[0], "кварталов ×", acc_mx.shape[1], "кварталов")
acc_mx.iloc[:5, :5].round(1)

In [ ]:
matrix_values = acc_mx.to_numpy(dtype=float)
inf_share = float(np.isinf(matrix_values).mean())
unreachable = np.isinf(matrix_values).any(axis=1)

finite_values = matrix_values[np.isfinite(matrix_values)]
print(f"Доля пар кварталов без маршрута: {inf_share:.2%}")
print(f"Кварталов, у которых хотя бы один маршрут отсутствует: {int(unreachable.sum())} из {len(acc_mx)}")
if finite_values.size:
    print(
        f"Время в пути по существующим маршрутам: медиана {np.median(finite_values):.1f} мин, "
        f"максимум {finite_values.max():.1f} мин"
    )

In [ ]:
from blocksnet.analysis.network.accessibility import mean_accessibility

# средняя доступность по достижимым кварталам (inf -> пропуск, чтобы не «портить» среднее)
mean_acc = mean_accessibility(acc_mx.replace([np.inf, -np.inf], np.nan))
mean_acc.index.name = "block_id"
save_df(mean_acc, "mean_accessibility_walk.csv")

ax = blocks.plot(color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
blocks.join(mean_acc).plot(
    column="mean_accessibility", ax=ax, cmap="RdYlGn_r", legend=True,
    missing_kwds={"color": "#d9d9d9"},
)
ax.set_title("Средняя доступность квартала, минуты (краснее — хуже)")
ax.set_axis_off()
plt.show()

if unreachable.any():
    print("Кварталы без части маршрутов (показаны на карте ниже):")
    ax = blocks.plot(color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
    blocks[unreachable].plot(ax=ax, color="#d62828", edgecolor="#7a1f1f", linewidth=0.4)
    ax.set_title("Кварталы, до которых нет части пешеходных маршрутов")
    ax.set_axis_off()
    plt.show()

## 8. Здания и население

Скачиваем здания из OSM, определяем жилые дома, число этажей, затем силами библиотеки восстанавливаем («импутируем») недостающие характеристики: площадь застройки, площадь этажей, жилую площадь и **население**.

Население — это **оценка**: она получается из площади жилых помещений и норматива 30 м² на человека (параметр `DEFAULT_LIVING_DEMAND`). Для учебного исследования этого достаточно, но с официальными данными результат может отличаться.

In [ ]:
from blocksnet.preprocessing.imputing import impute_buildings

IS_LIVING_TAGS = ["residential", "house", "apartments", "detached", "terrace", "dormitory"]
BUILDINGS_COLUMNS = [
    "geometry", "building", "is_living", "number_of_floors",
    "footprint_area", "build_floor_area", "living_area", "non_living_area", "population",
]

if has_file("buildings.geojson"):
    buildings = load_gdf("buildings.geojson")
    print(f"Здания прочитаны из файла: {len(buildings)}")
else:
    print("Скачиваем здания из OpenStreetMap...")
    polygon_4326 = boundaries.to_crs(4326).union_all()
    try:
        buildings = ox.features_from_polygon(polygon_4326, tags={"building": True})
    except Exception as error:  # обычно это сеть или пустой ответ OSM
        raise RuntimeError(
            f"Не удалось скачать здания из OSM: {error}. Проверьте интернет и запустите ячейку снова."
        ) from error
    buildings = buildings.reset_index(drop=True).to_crs(boundaries.crs)

    def is_living_building(tag):
        # у одного объекта OSM тег может быть списком значений
        values = tag if isinstance(tag, (list, tuple)) else [tag]
        return any(str(value).lower() in IS_LIVING_TAGS for value in values)

    buildings["is_living"] = buildings["building"].apply(is_living_building)
    if "building:levels" in buildings.columns:
        levels = buildings["building:levels"]
    else:
        levels = pd.Series(np.nan, index=buildings.index)
    buildings["number_of_floors"] = pd.to_numeric(levels, errors="coerce")
    buildings = impute_buildings(buildings, default_living_demand=DEFAULT_LIVING_DEMAND)
    buildings = buildings[[column for column in BUILDINGS_COLUMNS if column in buildings.columns]]
    save_gdf(buildings, "buildings.geojson")

print(f"Всего зданий: {len(buildings)}")
print(f"Жилых зданий: {int(buildings['is_living'].sum())}")
print(f"Оценка населения территории: {buildings['population'].sum():,.0f} чел.")
buildings.head(3)

In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects

if has_file("buildings_blocks.geojson"):
    buildings_blocks = load_gdf("buildings_blocks.geojson")
    print(f"Данные зданий по кварталам прочитаны из файла: {len(buildings_blocks)} кварталов")
else:
    print("Агрегируем здания по кварталам...")
    buildings_blocks, excess_buildings = aggregate_objects(blocks, buildings.copy())
    buildings_blocks.index.name = "block_id"
    save_gdf(buildings_blocks, "buildings_blocks.geojson")
    print(f"Зданий вне кварталов: {len(excess_buildings)}")

buildings_blocks.index.name = "block_id"
print(f"Население в границах территории: {buildings_blocks['population'].sum():,.0f} чел.")
print(f"Кварталов без зданий: {int((buildings_blocks['population'] <= 0).sum())}")

ax = blocks.plot(color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
buildings_blocks.plot(column="population", ax=ax, cmap="YlOrRd", legend=True)
ax.set_title("Население по кварталам, чел.")
ax.set_axis_off()
plt.show()

## 9. Сервисы: школы

Скачиваем школы из OSM. Ёмкость (число мест) в OSM обычно отсутствует — библиотека подставляет минимальную ёмкость из нормативов (`impute_services`), а затем считает **ёмкость по кварталам**.

Чтобы исследовать другие сервисы (детские сады, поликлиники, аптеки и т.д.), достаточно поменять `SERVICE_TYPE` и теги загрузки — теги для каждого сервиса можно уточнить вместе со мной.

In [ ]:
from blocksnet.config import service_types_config
from blocksnet.preprocessing.imputing import impute_services

SERVICE_TYPE = "school"  # тип сервиса из конфигурации blocksnet
SERVICE_TAGS = {"amenity": SERVICE_TYPE}
SERVICE_FILE = f"{SERVICE_TYPE}s.geojson"
SERVICE_BLOCKS_FILE = f"{SERVICE_TYPE}s_blocks.geojson"

service_config = service_types_config[SERVICE_TYPE]
print(f"Сервис {SERVICE_TYPE!r} ({service_config['name_ru']}): норматив обеспеченности "
      f"{service_config['demand']} мест на 1000 жителей, нормативная доступность {service_config['accessibility']} мин")

if has_file(SERVICE_FILE):
    services = load_gdf(SERVICE_FILE)
    print(f"Объекты прочитаны из файла {SERVICE_FILE}: {len(services)}")
else:
    print("Скачиваем объекты из OpenStreetMap...")
    polygon_4326 = boundaries.to_crs(4326).union_all()
    try:
        services = ox.features_from_polygon(polygon_4326, tags=SERVICE_TAGS)
    except Exception as error:  # обычно это сеть или пустой ответ OSM
        raise RuntimeError(
            f"Не удалось скачать объекты {SERVICE_TAGS} из OSM: {error}. "
            "Проверьте интернет и теги или увеличьте территорию."
        ) from error
    if len(services) == 0:
        raise RuntimeError("В OSM не найдено объектов этого типа — поменяйте SERVICE_TAGS или добавьте данные вручную.")
    services = services.reset_index(drop=True).to_crs(boundaries.crs)
    if "capacity" not in services.columns:
        services["capacity"] = np.nan  # обязательный столбец для impute_services в blocksnet 1.0.0a9
    services = impute_services(services, SERVICE_TYPE)
    services = services[[column for column in ["geometry", "name", "amenity", "capacity"] if column in services.columns]]
    save_gdf(services, SERVICE_FILE)

print(f"Объектов: {len(services)} | суммарная ёмкость: {services['capacity'].sum():,.0f} мест")
services.head(3)

In [ ]:
if has_file(SERVICE_BLOCKS_FILE):
    services_blocks = load_gdf(SERVICE_BLOCKS_FILE)
    print(f"Ёмкость по кварталам прочитана из файла {SERVICE_BLOCKS_FILE}")
else:
    services_blocks, excess_services = aggregate_objects(blocks, services.copy())
    services_blocks.index.name = "block_id"
    save_gdf(services_blocks, SERVICE_BLOCKS_FILE)
    print(f"Объектов вне кварталов: {len(excess_services)}")

services_blocks.index.name = "block_id"
print(f"Ёмкость по кварталам, всего: {services_blocks['capacity'].sum():,.0f}")
print(f"Кварталов с объектами этого типа: {int((services_blocks['capacity'] > 0).sum())} из {len(services_blocks)}")

ax = blocks.plot(color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
services_blocks.plot(column="capacity", ax=ax, cmap="Blues", legend=True)
ax.set_title(f"Ёмкость ({service_config['name_ru']}) по кварталам")
ax.set_axis_off()
plt.show()

## 10. Показатели

### 10.1. Плотностные показатели (FSI, GSI, MXI, L, OSR)

* **FSI** — коэффициент плотности застройки (площадь этажей / площадь участка);
* **GSI** — коэффициент застроенности (площадь застройки / площадь участка);
* **MXI** — доля жилых площадей;
* **L** — средняя этажность;
* **OSR** — обеспеченность открытыми пространствами.

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

if has_file("density_indicators.csv"):
    density_df = load_df("density_indicators.csv")
    print("Показатели плотности прочитаны из файла")
else:
    buildings_blocks["site_area"] = buildings_blocks.area
    density_df = calculate_density_indicators(buildings_blocks)
    save_df(density_df, "density_indicators.csv")

print("Показатели плотности:")
density_df[["fsi", "gsi", "mxi", "l", "osr"]].describe().round(2)

**Важно:** у кварталов без застройки (лес, вода, пустыри) часть показателей не определена: `MXI` и `L` будут пустыми, `OSR` может быть бесконечным — это нормально и не ошибка.

### 10.2. Морфотипы застройки

**Морфотип** — тип городской среды, определённый по сочетанию этажности, плотности и доли жилья. Считаем два подхода:

* **Spacematrix** — кластеризация кварталов (тип + кластер);
* **Strelka** — тип по правилам (низко-/средне-/высотная застройка и её характер).

In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

if has_file("spacematrix_morphotypes.csv") and has_file("spacematrix_clusters.csv"):
    spacematrix_df = load_df("spacematrix_morphotypes.csv")
    spacematrix_clusters = load_df("spacematrix_clusters.csv")
    print("Морфотипы Spacematrix прочитаны из файлов")
else:
    developed = int((density_df["fsi"] > 0).sum())
    n_clusters = max(2, min(11, developed))  # кластеров не больше, чем застроенных кварталов
    print(f"Застроенных кварталов: {developed} -> число кластеров: {n_clusters}")
    spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df, n_clusters=n_clusters)
    save_df(spacematrix_df, "spacematrix_morphotypes.csv")
    save_df(spacematrix_clusters, "spacematrix_clusters.csv")

if has_file("strelka_morphotypes.csv"):
    strelka_df = load_df("strelka_morphotypes.csv")
    print("Морфотипы Strelka прочитаны из файла")
else:
    strelka_df = get_strelka_morphotypes(density_df)
    save_df(strelka_df, "strelka_morphotypes.csv")

print("Кластеры Spacematrix (медианные значения и морфотип):")
spacematrix_clusters

In [ ]:
print("Морфотипы Spacematrix (кварталов):")
print(spacematrix_df["morphotype"].fillna("не определён").value_counts().to_string())
print()
print("Морфотипы Strelka (кварталов):")
print(strelka_df["morphotype"].fillna("не определён").value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
blocks.plot(ax=axes[0], color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3)
blocks.join(spacematrix_df[["morphotype"]]).plot(
    column="morphotype", categorical=True, legend=True, ax=axes[0], cmap="tab20",
    missing_kwds={"color": "#d9d9d9"},
)
axes[0].set_title("Морфотипы Spacematrix")
axes[0].set_axis_off()

blocks.plot(ax=axes[1], color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3)
blocks.join(strelka_df[["morphotype"]]).plot(
    column="morphotype", categorical=True, legend=True, ax=axes[1], cmap="tab20",
    missing_kwds={"color": "#d9d9d9"},
)
axes[1].set_title("Морфотипы Strelka")
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

### 10.3. Обеспеченность школами (конкурентная обеспеченность)

Считаем, насколько спрос квартала покрывается ёмкостью школ с учётом конкуренции между кварталами и нормативного времени доступности (из конфигурации blocksnet).

* спрос квартала = население, делённое на 1000, умноженное на норматив `demand` (мест на 1000 жителей), с округлением вверх;
* `provision_strong` — обеспеченность «внутри» нормативного времени: сколько спроса закрыто учреждениями, реально доступными жителям;
* `provision_weak` — более мягкая оценка, учитывающая все связанные ёмкости.

Значение **1.0** означает полное покрытие, меньше 1.0 — дефицит. В кварталах без населения спрос равен нулю, поэтому обеспеченность там не определяется (пустое значение) — это нормально.

In [ ]:
from blocksnet.analysis.provision import competitive_provision

demand_normative = int(service_config["demand"])  # норматив: мест на 1000 жителей
accessibility_minutes = int(service_config["accessibility"])  # нормативное время доступности, минуты

if has_file("provision_school.csv"):
    prov_df = load_df("provision_school.csv")
    links_df = pd.read_csv(DATA_DIR / "links_school.csv", index_col=[0, 1])
    print("Обеспеченность прочитана из файлов")
else:
    provision_input = buildings_blocks[["population"]].join(services_blocks[["capacity"]])
    print("Решаем задачу распределения спроса по ёмкостям (может занять минуту)...")
    prov_df, links_df = competitive_provision(
        provision_input, acc_mx, accessibility=accessibility_minutes, demand=demand_normative
    )
    save_df(prov_df, "provision_school.csv")
    links_df.to_csv(DATA_DIR / "links_school.csv")
    print("[сохранено] links_school.csv")

print("Сводка по обеспеченности:")
print(prov_df[["provision_strong", "provision_weak"]].describe().round(2).to_string())

ax = blocks.plot(color="#e6e6e6", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
blocks.join(prov_df[["provision_strong"]]).plot(
    column="provision_strong", ax=ax, cmap="RdYlGn", vmin=0, vmax=1, legend=True,
    missing_kwds={"color": "#f0f0f0"},
)
ax.set_title("Обеспеченность школами (provision_strong): 1 — норма, 0 — дефицит")
ax.set_axis_off()
plt.show()

### 10.4. Доступность территории

`area_accessibility` — средневзвешенное (по площади кварталов) время доступности: показывает, насколько «в среднем» территория связана сама с собой. Чем меньше значение, тем компактнее и связнее территория.

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

if has_file("area_accessibility.csv"):
    area_acc = load_df("area_accessibility.csv")
    print("Доступность территории прочитана из файла")
else:
    area_acc = area_accessibility(acc_mx, buildings_blocks)
    save_df(area_acc, "area_accessibility.csv")

area_mean = area_acc["area_accessibility"].replace([np.inf, -np.inf], np.nan).mean()
print(f"Средняя доступность территории: {area_mean:.2f} мин")

ax = blocks.plot(color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3, figsize=(9, 9))
blocks.join(area_acc).plot(column="area_accessibility", ax=ax, cmap="RdYlGn_r", legend=True)
ax.set_title("Доступность квартала, минуты")
ax.set_axis_off()
plt.show()

### 10.5. Итоговый слой результатов

Собираем всё в один слой **«квартал → показатели»** и сохраняем в `data/blocks_indicators.geojson`. Именно этот файл (и CSV с показателями) будет основой интерактивной карты и дашборда на следующем этапе.

In [ ]:
INDICATOR_COLUMNS = [
    "land_use", "share", "population", "site_area",
    "fsi", "gsi", "mxi", "l", "osr",
    "spacematrix_cluster", "morphotype_spacematrix", "morphotype_strelka",
    "demand", "capacity", "provision_strong", "provision_weak",
    "area_accessibility", "mean_accessibility",
]

blocks_indicators = blocks_land_use.copy()
blocks_indicators = blocks_indicators.join(buildings_blocks[["population"]])
blocks_indicators = blocks_indicators.join(density_df[["fsi", "gsi", "mxi", "l", "osr"]])
blocks_indicators = blocks_indicators.join(
    spacematrix_df[["cluster", "morphotype"]].rename(
        columns={"cluster": "spacematrix_cluster", "morphotype": "morphotype_spacematrix"}
    )
)
blocks_indicators = blocks_indicators.join(
    strelka_df[["morphotype"]].rename(columns={"morphotype": "morphotype_strelka"})
)
blocks_indicators = blocks_indicators.join(prov_df[["demand", "capacity", "provision_strong", "provision_weak"]])
blocks_indicators = blocks_indicators.join(area_acc)
blocks_indicators = blocks_indicators.join(mean_acc)
blocks_indicators.index.name = "block_id"

if has_file("blocks_indicators.geojson"):
    print("Итоговый слой уже сохранён — обновляем его свежими расчётами")
save_gdf(blocks_indicators, "blocks_indicators.geojson")

available = [column for column in INDICATOR_COLUMNS if column in blocks_indicators.columns]
print("Столбцов в итоговом слое:", len(blocks_indicators.columns))
blocks_indicators[available].head(5)

In [ ]:
numeric_summary = [
    "population", "fsi", "gsi", "mxi", "l", "osr",
    "provision_strong", "provision_weak", "area_accessibility", "mean_accessibility",
]
print("Сводная статистика показателей по кварталам:")
print(blocks_indicators[numeric_summary].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16), sharex=False, sharey=False)

blocks_indicators.plot(column="fsi", ax=axes[0, 0], cmap="viridis", legend=True)
axes[0, 0].set_title("FSI — плотность застройки")

blocks_indicators.plot(column="mxi", ax=axes[0, 1], cmap="magma", legend=True)
axes[0, 1].set_title("MXI — доля жилых площадей")

blocks_indicators.plot(column="l", ax=axes[1, 0], cmap="cividis", legend=True)
axes[1, 0].set_title("L — средняя этажность")

blocks_indicators.plot(column="mean_accessibility", ax=axes[1, 1], cmap="RdYlGn_r", legend=True)
axes[1, 1].set_title("Средняя доступность, минуты")

for axis in axes.ravel():
    axis.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

use_counts.plot(kind="barh", ax=axes[0], color="#4a7fb5")
axes[0].set_title("Кварталы по преобладающему land use")
axes[0].set_xlabel("кварталов")

blocks_indicators["provision_strong"].dropna().clip(0, 1).plot(
    kind="hist", bins=20, ax=axes[1], color="#e07a5f", edgecolor="white"
)
axes[1].set_title("Распределение обеспеченности школами (provision_strong)")
axes[1].set_xlabel("обеспеченность, 1 = норма")

plt.tight_layout()
plt.show()

## 11. Проверка результатов

Здесь собраны простые проверки: они печатают `[ОК]` или `[ПРОБЛЕМА]` с пояснением. Если всё `[ОК]`, результаты можно использовать для дальнейшего анализа и для дашборда.

In [ ]:
EXPECTED_FILES = [
    "boundaries.geojson", "water.geojson", "roads.geojson", "railways.geojson",
    "lines.geojson", "polygons.geojson", "functional_zones.geojson",
    "blocks.geojson", "blocks_land_use.geojson",
    "walk_graph.pkl", "walk_nodes.geojson", "walk_edges.geojson",
    "accessibility_matrix_walk.csv", "mean_accessibility_walk.csv",
    "buildings.geojson", "buildings_blocks.geojson", "schools.geojson", "schools_blocks.geojson",
    "density_indicators.csv", "spacematrix_morphotypes.csv", "spacematrix_clusters.csv",
    "strelka_morphotypes.csv", "provision_school.csv", "links_school.csv",
    "area_accessibility.csv", "blocks_indicators.geojson",
]

missing_files = [name for name in EXPECTED_FILES if not has_file(name)]
land_use_filled = float(blocks_land_use["land_use"].notna().mean())
population_total = float(buildings_blocks["population"].sum())
capacity_total = float(services_blocks["capacity"].sum())
density_ok = bool(
    density_df[["fsi", "gsi"]].notna().all().all()
    and np.isfinite(density_df[["fsi", "gsi"]].to_numpy(dtype=float)).all()
)
provision_ok = bool(prov_df.loc[prov_df["demand"] > 0, "provision_strong"].notna().all())

check("Кварталы нарезаны", len(blocks) > 0, f"кварталов: {len(blocks)}")
check("CRS в метрах (проекционная)", boundaries.crs is not None and boundaries.crs.is_projected,
      f"EPSG:{boundaries.crs.to_epsg() if boundaries.crs else 'нет'}")
check("Площадь кварталов положительна", blocks.area.sum() > 0, f"{blocks.area.sum() / 1e6:.1f} км²")
check("Матрица совпадает с кварталами", list(acc_mx.index) == list(blocks.index) and acc_mx.shape[0] == acc_mx.shape[1],
      f"{acc_mx.shape[0]} × {acc_mx.shape[1]}")
check("Пешеходные маршруты есть почти везде (доля inf < 10%)", inf_share < 0.10, f"доля inf: {inf_share:.2%}")
check("Land use определён у большинства кварталов", land_use_filled > 0.5, f"{land_use_filled:.0%} кварталов")
check("Население посчитано", population_total > 0, f"{population_total:,.0f} чел. (оценка)")
check("Ёмкость школ посчитана", capacity_total > 0, f"{capacity_total:,.0f} мест")
check("GSI в диапазоне 0…1", bool(density_df["gsi"].dropna().between(0, 1).all()), "коэффициент застроенности")
check("Показатели плотности посчитаны", density_ok, "FSI и GSI есть у всех кварталов")
check("Обеспеченность посчитана", provision_ok, "там, где есть население")
check("Индексы итогового слоя совпадают с кварталами", blocks_indicators.index.equals(blocks.index))
check("Все файлы результатов на месте", len(missing_files) == 0, f"нет файлов: {missing_files}")

**Что делать, если появилась «[ПРОБЛЕМА]»**

| Проверка | Что обычно означает | Что делать |
|---|---|---|
| Кварталы не нарезаны | пустая граница или мало данных OSM | проверить `boundaries.geojson` на карте, расширить границу (свой файл в `data/boundaries.geojson`) |
| Много `inf` в матрице | часть кварталов отрезана (река, нет мостов, край территории) | посмотреть карту «кварталы без маршрутов»; при необходимости расширить территорию |
| Land use определён у меньшей части кварталов | в OSM нет `landuse` для этих участков | нормально для окраин: это не ошибка, но стоит упомянуть в выводах |
| Население = 0 | не загрузились здания или нет тега `building` | посмотреть `data/buildings.geojson`, проверить интернет и кеш |
| Нет файлов результатов | ячейка расчётов не выполнялась или упала с ошибкой | запустить раздел заново сверху вниз |

In [ ]:
files_table = list_data_files()
files_table

## 12. Повторный запуск: инструкция

### Порядок запуска

1. Откройте ноутбук `examples/gusev/gusev.ipynb` в VS Code.
2. Справа сверху выберите ядро (kernel) **`.venv`** — то же окружение, где установлены `blocksnet`, `osmnx`, `iduedu`.
3. Нажмите **«Запустить все» / Run All** (или выполняйте ячейки по порядку сверху вниз).

### Сколько это занимает

| Ситуация | Время |
|---|---|
| Первый запуск (скачивание объектов, зданий и графа для города ≈ 21 км²) | обычно 3–10 минут |
| Повторный запуск (всё читается из `data/`) | меньше минуты |
| Расчёт матрицы времени в пути | секунды — несколько минут |

### Если нужно скачать данные заново

В ячейке с функциями есть `clear_cache()`:

* `clear_cache()` — удалить все сохранённые файлы (данные скачаются заново);
* `clear_cache(include_osm_cache=False)` — удалить результаты, но сохранить локальный кеш OSM.

Удалить только один шаг можно вручную: например, удалите `data/walk_graph.pkl`, и при следующем запуске скачается только граф.

### Если что-то не работает

* **«Не удалось скачать / ошибка сети»** — проверьте интернет и просто запустите ячейку ещё раз: часть данных уже сохранена, повторная загрузка идёт быстрее.
* **Скачивание графа или зданий идёт слишком долго / падает по таймауту** — так бывает при перегрузке серверов OSM: просто запустите ячейку ещё раз, часть данных уже сохранена. Если это повторяется, удалите файлы `data/boundaries.geojson`, `data/blocks*.geojson`, `data/functional_zones.geojson`, `data/water/roads/railways.geojson`, `data/*graph*`, `data/*accessibility*`, `data/buildings*.geojson` и запустите ноутбук заново.
* **Непонятная ошибка в расчётной ячейке** — удалите файл этого шага из `data/` и выполните раздел заново.
* **Нужно расширить или сузить территорию** — граница берётся из OSM по `OSM_RELATION_ID`. Чтобы взять другую территорию, положите свой полигон в `data/boundaries.geojson` (в той же системе координат, что и остальные файлы, метры) и удалите перечисленные выше файлы, чтобы они пересчитались для новой границы.

### Что дальше

Следующий (отдельный) этап — Python-файл, который соберёт **одну HTML-страницу**: интерактивную карту кварталов с показателями и дашборд с графиками, таблицами и выводами. Основой будут файлы `data/blocks_indicators.geojson`, `data/accessibility_matrix_walk.csv`, `data/provision_school.csv`, `data/links_school.csv` и другие из папки `data`.